In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pickle
# disable warnings
import warnings
warnings.filterwarnings('ignore')

# import functions from my_path\utils.py
# use absolute path to import functions
import sys
sys.path.append(r'C:/Users/Zheng/OneDrive - UW-Madison/PhD project/AARC/phd work major comparison')
from utils import *

In [ ]:
df_dist = pd.read_csv(
    r"my_path\10 interdiscipnarity factor\distance factor\discipline_distance_cross.csv"
)
# remove humanities
df_dist = df_dist.query("phd_community != 5")

# import productivity data
df_doi_info = pd.read_csv(
    r"my_path\7. get productivity\person_doi_info.csv"
)

# restrict number of authors
df_doi_info = df_doi_info.query("Nb_Auteur <= 10")

# import IDR data and degree year
df_idr = pd.read_csv(r"discipline_idr_bin.csv")

# institution and degree year
df_inst = pd.read_csv(
    r"my_path\university match\faculty_inst_phdinst.csv"
)

# a WOS field's probability of being in a certain AA area
# each WOS field has a vector which sums to 1
df_wos2aa = pd.read_csv(
    r"my_path\7. get productivity\aa2wos_area_prob.csv"
)

# read gender data
df_gender = pd.read_csv(
    r"my_path\5. get gender and race\gender_match_discipline.csv"
)
display(df_gender["gender"].value_counts())

gender
M    31999
F    24358
U     2172
Name: count, dtype: int64

In [ ]:
# read univ rank
df_univ = pd.read_csv(
    r"my_path\6. get dept rank\institution_rank_by_area.csv"
)
# change area names in df_univ
# replace full names with short names
df_univ["area"] = df_univ["area"].map(abbr)
display(df_univ)

,Rank,InstitutionId,Percentile,area
0,0.000000,7,0.000000,Sociology
1,0.250711,217,0.440529,Sociology
2,0.547605,539130,0.881057,Sociology
3,0.713267,395,1.321586,Sociology
4,0.714773,388,1.762115,Sociology
...,...,...,...,...
4956,10.501938,49,98.181818,Physics
4957,10.963708,198,98.636364,Physics
4958,11.083249,251,99.090909,Physics
4959,11.207247,167,99.545455,Physics


# productivity

In [ ]:
# get each person's life time pub
df_wos_pub = (
    df_dist
    .merge(
        df_inst[["PersonId", "DegreeYear"]], on="PersonId", how="inner"
    )
    .merge(
        df_doi_info[["PersonId", "OST_BK", "Annee_Bibliographique", "Code_Discipline", "Nb_Auteur"]],
        on="PersonId",
        how="inner",
    )
    # merge idr
    .merge(
        df_idr[["PersonId", "rao", 'rao_bin']], on="PersonId", how="inner"
    )
    .drop_duplicates()
)

# add tau
df_wos_pub["tau"] = df_wos_pub["Annee_Bibliographique"] - df_wos_pub["DegreeYear"]

In [ ]:
# filter to PhD works and count unique OST_BK
df_prod_total = (
    df_wos_pub.query("tau <= 1 and tau >= -4")
    .groupby(["PersonId"])
    .agg(n=("OST_BK", "nunique"))
    .reset_index()
)

# cumulative percentage of people by n
t = (
    df_prod_total.groupby("n")["PersonId"].count().cumsum()
    / df_prod_total["PersonId"].count()
).reset_index()

t.columns = ["# papers", "cumulative % of PhDs"]

# plot
fig = px.line(t, x="# papers", y="cumulative % of PhDs")
plot_styling(fig, size=(600, 600))
fig

In [16]:
# remove very extreme n
df_prod_total = df_prod_total.query("n >= 1 and n <= 20")

# make an idlist
idlist = df_prod_total["PersonId"].unique()
# save idlist
# with open("personid_list_2to21.pkl", "wb") as f:
#     pickle.dump(idlist, f)

# remove from df_wos_pub
df_wos_pub = df_wos_pub.query("PersonId in @idlist")

In [17]:
# regression

import stata_setup

stata_setup.config("C:/Program Files/Stata17/", "se")
from pystata import stata

In [ ]:
# year by year productivity, including after PhD
groupvar = "DegreeYear"
groupvar2, group2 = "phd_community", list(community_names.keys())


def fill_nan_with_zero(val, ind, col):
    if pd.isna(val):
        person_id = ind
        tau_range = t_range[t_range["PersonId"] == person_id]
        if not tau_range.empty:
            min_tau = tau_range["min_tau"].values[0]
            max_tau = tau_range["max_tau"].values[0]
            if min_tau <= col <= max_tau:
                return 0
    return val


t = df_wos_pub.query("tau >= -4 and tau <= 10").query(
    "DegreeYear >= 2005 and DegreeYear <= 2018"
)
t["all"] = 1
t = t[["PersonId", "tau", "DegreeYear", "OST_BK", groupvar2]].drop_duplicates()
tau_range = np.arange(-4, 11)

# get min and max tau
t["min_tau"] = t["DegreeYear"].apply(lambda x: max(tau_range[0], 2000 - x))
t["max_tau"] = t["DegreeYear"].apply(lambda x: min(tau_range[-1], 2021 - x))
t_range = t[["PersonId", "min_tau", "max_tau"]].drop_duplicates()

df_cite_norm = (
    t.groupby([groupvar, groupvar2, "PersonId", "tau"])
    .agg(n=("OST_BK", "nunique"))
    .reset_index()
    .query("tau in @tau_range")
    .pivot(columns=["tau"], values=["n"], index=[groupvar, groupvar2, "PersonId"])
)
# fill 0 for tau range
for ind, row in df_cite_norm.iterrows():
    for col in df_cite_norm.columns:
        val = row[col]
        row[col] = fill_nan_with_zero(val, ind[-1], col[-1])

df_cite_norm = df_cite_norm.stack().reset_index()

df_cite_norm

In [ ]:
# by quintile
r= 10
df_norm = (df_cite_norm
    # merge with idr
    .merge(
        df_idr[["PersonId", "rao", 'rao_bin']], on="PersonId", how="inner"
    )
    # field
    .merge(df_dist[['PersonId', 'prof_area', 'phd_area']], on="PersonId", how="inner")
    # gender
    .merge(df_gender.query("gender != 'U'"), on="PersonId", how="inner")
    # get university rank
    .merge(
        df_inst[["PersonId", "InstitutionId"]], on="PersonId", how="inner"
    )
    .merge(
        df_univ.rename(
            columns={
                "area": "prof_area",
            }
        )[["InstitutionId", "Percentile", 'prof_area']],
        on=["InstitutionId", "prof_area"],
        how="inner",
    )
    # mark top 10% univ
    .assign(top=lambda x: (x["Percentile"] >= 100 - r).astype(int))
    # merge with get degree university rank
    .merge(
        df_inst[["PersonId", "DegreeInstitutionID"]], on="PersonId", how="inner"
    )
    .merge(
        df_univ.rename(
            columns={
                "InstitutionId": "DegreeInstitutionID",
                "Percentile": "DegreePercentile",
                "area": "phd_area",
            }
        )[["DegreeInstitutionID", "DegreePercentile", 'phd_area']],
        on=["DegreeInstitutionID", 'phd_area'],
        how="inner",
    )
    # get annee biblio
    .assign(Annee_Bibliographique=lambda x: x["tau"] + x["DegreeYear"])
    .drop_duplicates()
)

df_norm

In [ ]:
df_prod_cite = pd.read_csv("person_highest_cite_prod.csv").rename(columns={"n": "prod"})
df_advisor = pd.read_csv("advisor_attr.csv")
df_collab = pd.read_csv("collab_num.csv", usecols=["PersonId", "collab_num"])

r = 10

df_norm = (df_cite_norm
    # merge with idr
    .merge(
        df_idr[["PersonId", "rao"]], on="PersonId", how="inner"
    )
    # field
    .merge(df_dist[['PersonId', 'prof_area', 'phd_area']], on="PersonId", how="inner")
    # gender
    .merge(df_gender.query("gender != 'U'"), on="PersonId", how="inner")
    .merge(df_prod_cite, on="PersonId", how="inner")
    # advisor
    .merge(df_advisor, on="PersonId", how="inner")
    .merge(df_collab, on="PersonId", how="left")
    # get university rank
    .merge(
        df_inst[["PersonId", "InstitutionId"]], on="PersonId", how="inner"
    )
    .merge(
        df_univ.rename(
            columns={
                "area": "prof_area",
            }
        )[["InstitutionId", "Percentile", 'prof_area']],
        on=["InstitutionId", "prof_area"],
        how="inner",
    )
    # mark top 10% univ
    .assign(top=lambda x: (x["Percentile"] >= 100 - r).astype(int))
    # merge with get degree university rank
    .merge(
        df_inst[["PersonId", "DegreeInstitutionID"]], on="PersonId", how="inner"
    )
    .merge(
        df_univ.rename(
            columns={
                "InstitutionId": "DegreeInstitutionID",
                "Percentile": "DegreePercentile",
                "area": "phd_area",
            }
        )[["DegreeInstitutionID", "DegreePercentile", 'phd_area']],
        on=["DegreeInstitutionID", 'phd_area'],
        how="inner",
    )
    # get annee biblio
    .assign(Annee_Bibliographique=lambda x: x["tau"] + x["DegreeYear"])
    .fillna({"collab_num": 0, "cite_norm": 0})
    .drop_duplicates()
)

df_norm

In [ ]:
# analysis period
tau_range = np.arange(-4, 11)

t = df_norm.copy()

ts = []

for i, area in enumerate(list(community_names.keys())):
    t_area = t.query("phd_community == @area")
    t_area['tau'] = t_area['tau'] + 5
    
    # # standardize
    t_person = t_area[['PersonId', 'rao']].drop_duplicates()
    t_person['rao'] = (t_person['rao'] - t_person['rao'].mean()) / t_person['rao'].std()
    t_area = t_area.drop(columns="rao").merge(t_person, on='PersonId', how='left')
    
    print(f"Area: {area}")

    command = f"""
    encode(gender), gen(gender2)
    encode(adv_gender), gen(adv_gender2)
    qui poisson n i.tau##i.top##c.rao DegreePercentile gender2 adv_gender2 adv_n seniority i.Annee_Bibliographique
    margins tau, dydx(rao) at(top=(0 1))
    """
    
    #  i.tau c.rao i.top i.tau#c.rao i.tau#c.rao#i.top
    
    stata.run("clear")
    stata.pdataframe_to_data(t_area)
    stata.run(command)
    
    # collect results
    results = stata.get_return()["r(table)"].T
    results = pd.DataFrame(results).iloc[:, :6]
    results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
    results['is_top'] = [0] * len(tau_range) + [1] * len(tau_range)
    results['tau'] = [i for i in tau_range] * 2
    results["phd_community"] = area
    ts.append(results)
    
data = pd.concat(ts)

In [ ]:
# plot
groupvar, group = "phd_community", list(community_names.keys())
titles = [name for name in community_names.values()]

fig = make_subplots(
    rows=1,
    cols=5,
    subplot_titles=titles,
    vertical_spacing=0.15,
    horizontal_spacing=0.06,
)
names = ["Non-top", f"Top {r}%"]
colors = generate_gradient_colors(2)

for i, domain in enumerate(group):
    for j, top in enumerate(range(2)):

        t1_domain = data.query(f"{groupvar} == @domain and is_top == @top")
        t1 = t1_domain.dropna()

        x = tau_range
        y1 = t1["coef"].to_numpy()
        y1_upper, y1_lower = (
            t1["ci_low"].to_numpy(),
            t1["ci_high"].to_numpy(),
        )

        fig = add_lines_with_errorband(
            fig,
            x,
            y1,
            y1_upper,
            y1_lower,
            name=names[j],
            color=colors[j],
            showlegend=True if i == 0 else False,
            row=1,
            col=i % 5 + 1,
        )

fig = plot_styling(fig, size=(260, 900), title=None)
# update legend title
fig.update_layout(
    legend=dict(
        title="Faculty university rank",
        orientation="h",
        yanchor="top",
        y=-0.3,
        yref="paper",
        xanchor="center",
        x=0.5,
        xref="paper",
    )
)
fig.update_yaxes(
    title=dict(text="Productivity change<br>per 1 SD increase<br>in PhD interdisciplinarity"),
    titlefont=dict(color="black"),
    tickfont=dict(color="black"),
    col=1,
)

# zeroline
fig.add_vline(x=0, line=dict(dash="dash", width=1))
fig.add_hline(y=0, line=dict(dash="dash", width=1))

# Update layout to include super x and y labels
fig.update_xaxes(title=dict(text=None, standoff=0))
fig.add_annotation(
    text="Relative year to PhD graduation",
    x=0.5,
    y=-0.3,
    showarrow=False,
    xref="paper",
    xanchor="center",
    yref="paper",
    yanchor="bottom",
    font=dict(size=14),
)

fig.show()
fig.write_image(f"figures/prod_premium_top{r}.pdf")